In [65]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import trange
from tqdm.cli import tqdm

## prepocessing

In [66]:
df_belchonok = pd.read_csv('./preprocessed_data/belchonok.csv')
df_future = pd.read_csv('./preprocessed_data/future.csv')
df_granit = pd.read_csv('./preprocessed_data/granit_nauki.csv')
df_innopolis = pd.read_csv('./preprocessed_data/innopolis.csv')
df_itmo = pd.read_csv('./preprocessed_data/itmo.csv')
df_mosh = pd.read_csv('./preprocessed_data/mosh.csv')
df_open = pd.read_csv('./preprocessed_data/open.csv')
df_vuz = pd.read_csv('./preprocessed_data/vuz_akadem.csv')
df_mow_sch = pd.read_csv('./preprocessed_data/MOW_SCH.csv')
df_mow_mun = pd.read_csv('./preprocessed_data/MOW_MUN.csv')
df_reg = pd.read_csv('./preprocessed_data/reg.csv')
df_arso_fin = pd.read_csv('./preprocessed_data/fin.csv')

### Фильтрация по участникам

In [67]:
df_arso_fin = df_arso_fin.sort_values(by=['year', 'score'], ascending=[True, False])
df_arso_fin.head()

,place,name,region,class_study,p1,p2,p3,p4,p5,p6,p7,p8,score,title,year
0,1,Грекова Д.,Москва,11,100,100,100,70,100,88,86,80,724,1,24-25
1,2,Абдуллин Г.,Республика Татарстан,11,100,100,100,70,100,100,70,80,720,1,24-25
2,3,Жиганов В.,Москва,10,100,100,88,60,100,100,66,68,682,1,24-25
3,4,Звездин В.,Санкт-Петербург,10,100,100,100,70,100,76,66,64,676,1,24-25
4,5,Иванов Н.,Москва,11,100,100,88,65,100,100,65,50,668,1,24-25


#### Проблема с совпадением фамилия + первая буква имени

In [68]:
names = df_arso_fin['name'].value_counts().keys().to_numpy()

In [69]:
names[df_arso_fin['name'].value_counts() > 2]

array(['Макаров М.', 'Закиров А.', 'Торба А.', 'Андреев А.', 'Осипов Д.',
       'Зайцев А.', 'Соколов М.'], dtype=object)

In [70]:
df_arso_fin['name'].value_counts()

name
Макаров М.       4
Закиров А.       4
Торба А.         3
Андреев А.       3
Осипов Д.        3
                ..
Виноградов П.    1
Зуева А.         1
Кочуров Е.       1
Курзин И.        1
Яковлева П.      1
Name: count, Length: 719, dtype: int64

In [71]:
df_arso_fin[df_arso_fin['name'].isin(names[df_arso_fin['name'].value_counts() > 2])].sort_values(by=['place'], ascending=[True])

,place,name,region,class_study,p1,p2,p3,p4,p5,p6,p7,p8,score,title,year
505,13,Макаров М.,Москва,11,100,100,26,40,100,100,89,34,589,1.0,25-26
508,16,Макаров М.,Москва,11,100,100,20,31,100,95,100,25,571,1.0,25-26
28,29,Макаров М.,Москва,10,100,100,63,0,100,100,65,60,588,1,24-25
528,35,Закиров А.,Республика Татарстан,10,100,100,26,35,100,85,59,32,537,1.0,25-26
41,41,Закиров А.,Республика Татарстан,9,100,100,63,44,100,71,55,33,566,2,24-25
534,42,Закиров А.,Республика Татарстан,11,100,100,20,32,100,85,59,34,530,2.0,25-26
98,99,Торба А.,Москва,10,100,86,49,30,100,66,40,46,517,2,24-25
107,108,Закиров А.,Республика Татарстан,10,100,100,51,16,100,76,50,17,510,2,24-25
604,111,Торба А.,Москва,11,100,46,26,41,100,85,59,25,482,2.0,25-26
606,113,Андреев А.,Ростовская область,11,100,51,26,35,100,85,59,25,481,2.0,25-26


**РЕШЕНИЕ**:
слииив

In [72]:
df_arso_fin=df_arso_fin[df_arso_fin['year'] == '25-26']
df_arso_fin=df_arso_fin.drop(columns=['region', 'year', 'title']).groupby('name', as_index=False).mean()
df_arso_fin.sort_values(by=['place'], inplace=True)
df_arso_fin[10:30]

,name,place,class_study,p1,p2,p3,p4,p5,p6,p7,p8,score
141,Зайцев Д.,11.0,11.0,100.0,100.0,26.0,31.0,100.0,85.0,100.0,64.0,606.0
293,Назыров С.,12.0,11.0,100.0,100.0,26.0,21.0,100.0,100.0,77.0,67.0,591.0
218,Кузнецов И.,14.0,11.0,100.0,100.0,20.0,35.0,100.0,85.0,100.0,42.0,582.0
255,Макаров М.,14.5,11.0,100.0,100.0,23.0,35.5,100.0,97.5,94.5,29.5,580.0
438,Цыбань Л.,15.0,11.0,100.0,100.0,38.0,41.0,100.0,100.0,59.0,34.0,572.0
478,Янковский А.,17.0,10.0,100.0,31.0,73.0,37.0,100.0,85.0,77.0,67.0,570.0
274,Мельников И.,18.0,11.0,100.0,100.0,26.0,31.0,100.0,85.0,59.0,67.0,568.0
295,Насыбуллин И.,19.0,10.0,100.0,100.0,37.0,35.0,100.0,100.0,59.0,34.0,565.0
331,Пинский А.,20.0,7.0,100.0,100.0,20.0,31.0,100.0,85.0,83.0,45.0,564.0
29,Аширов М.,21.0,10.0,100.0,100.0,52.0,31.0,100.0,85.0,59.0,34.0,561.0


In [73]:
df_arso_fin

,name,place,class_study,p1,p2,p3,p4,p5,p6,p7,p8,score
149,Звездин В.,1.0,11.0,100.0,100.0,58.0,100.0,100.0,100.0,100.0,74.0,732.0
217,Кузнецов Е.,2.0,10.0,100.0,100.0,98.0,54.0,100.0,95.0,100.0,64.0,711.0
337,Приводнов М.,3.0,10.0,100.0,100.0,95.0,58.0,100.0,100.0,77.0,80.0,710.0
134,Жиганов В.,4.0,11.0,100.0,100.0,90.0,64.0,100.0,100.0,59.0,67.0,680.0
208,Кравченко М.,5.0,10.0,100.0,100.0,57.0,47.0,100.0,100.0,100.0,67.0,671.0
...,...,...,...,...,...,...,...,...,...,...,...,...
150,Зуева А.,482.0,11.0,100.0,11.0,13.0,0.0,0.0,15.0,0.0,6.0,145.0
206,Кочуров Е.,483.0,10.0,42.0,9.0,20.0,13.0,36.0,0.0,10.0,0.0,130.0
86,Голов С.,484.0,11.0,100.0,9.0,0.0,0.0,0.0,15.0,0.0,0.0,124.0
230,Курзин И.,485.0,9.0,36.0,9.0,13.0,0.0,36.0,10.0,10.0,6.0,120.0


In [74]:
dfs = [df_belchonok, df_granit, df_innopolis, df_itmo, df_mosh, df_open, df_vuz, df_mow_sch, df_mow_mun, df_reg, df_future]

In [75]:
for i in range(len(dfs) - 1):
    j = dfs[i]
    num_cols = j.select_dtypes(include='number').columns.tolist()
    j[num_cols] = j[num_cols].fillna(0)
    j = j[j['name'].isin(df_arso_fin['name'].to_numpy())]
    j = j[j['year'] == '25-26']
    dfs[i] = j
dfs[10] = dfs[10][dfs[10]['name'].isin(df_arso_fin['name'].to_numpy())]

In [76]:
dfs[5]

,name,grade,participation_grade,reg,A1,B1,C1,D1,A2,B2,C2,D2,total,result,year
429,Кузнецов И.,11,11,Санкт-Петербург,30.0,100.0,100.0,100.0,95.0,100.0,100.0,100.0,725,3,25-26
430,Звездин В.,11,11,Санкт-петербург,30.0,100.0,100.0,100.0,84.0,100.0,100.0,100.0,714,3,25-26
431,Муханов С.,11,11,Москва,30.0,100.0,82.0,100.0,91.0,100.0,100.0,100.0,703,3,25-26
433,Жиганов В.,11,11,Москва,30.0,100.0,100.0,100.0,98.0,100.0,100.0,50.0,678,3,25-26
434,Гришко Д.,11,11,Москва,30.0,100.0,100.0,100.0,64.0,100.0,100.0,35.0,629,3,25-26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
870,Кувардина Ю.,11,11,Москва,0.0,100.0,9.0,0.0,0.0,0.0,0.0,0.0,109,0,25-26
871,Трухачев Ф.,10,11,Москва,7.0,79.0,0.0,23.0,0.0,0.0,0.0,0.0,109,0,25-26
873,Гилёв Г.,9,11,Москва,7.0,31.0,9.0,0.0,13.0,9.0,30.0,9.0,108,0,25-26
874,Иванов П.,11,11,Москва,7.0,63.0,19.0,14.0,0.0,0.0,0.0,0.0,103,0,25-26


In [77]:
l = dfs[9].columns[3:-2]
for i in l:
    j = dfs[9][i].astype(float)
    j = j.fillna(0)
    dfs[9][i] = j
    dfs[9].rename({i: i + '1'})
dfs[9]

,name,region,grade,A1,B1,C1,D1,A2,B2,C2,D2,Sum,year
7833,Гриненко А.,Москва,10,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,800.0,25-26
7834,Звездин В.,Санкт-Петербург,11,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,800.0,25-26
7835,Кравченко М.,Москва,10,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,800.0,25-26
7836,Кузнецов И.,Санкт-Петербург,11,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,800.0,25-26
7837,Кузнецов Е.,Липецкая область,10,100.0,88.0,100.0,100.0,100.0,100.0,100.0,55.0,743.0,25-26
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13081,Васильев А.,Свердловская область,9,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,25-26
13130,Иванов П.,Владимирская область,11,0.0,0.0,0.0,0.0,5.0,0.0,6.0,0.0,11.0,25-26
13268,Васильев М.,Донецкая Народная Республика,9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25-26
13275,Волков А.,Ярославская область,11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25-26


Не смотрите в эти макароны - опасно, возможен вывих глаза

In [78]:
alll = dfs[0]
alll = pd.merge(alll, dfs[1], on='name', how='outer')
alll = pd.merge(alll, dfs[2], on='name', how='outer').drop(columns=['result', 'reg_y', 'reg_x', 'year', 'year_y', 'year_x']).rename(columns={'total_x': 'total1', 'total_y': 'total2', 'grade_x': 'grade1', 'grade_y': 'grade2', 'result_x': 'result1', 'result_y': 'result2'})
alll = pd.merge(alll, dfs[3], on='name', how='outer').drop(columns=['year']).rename(columns={'score': 'score1'})
alll = pd.merge(alll, dfs[4].drop(columns=['class_study']), on='name', how='outer').drop(columns=['id', 'year',  'title', 'reg']).rename(columns={'result': 'result3', 'grade': 'grade3', 'total': 'total3'})
alll = pd.merge(alll, dfs[5], on='name', how='outer').drop(columns=['reg', 'year']).rename(columns={'result': 'result4', 'grade': 'grade4', 'total': 'total4', 'score': 'score4'})
alll = pd.merge(alll, dfs[6], on='name', how='outer').drop(columns=['reg', 'year', 'region', 'class_partake'])
alll = pd.merge(alll, dfs[7].drop(columns=['class_study']), on='name', how='outer').drop(columns=['id', 'year', 'class_partake', 'region', 'title', 'participation_grade']).rename(columns={'result': 'result5', 'grade': 'grade5', 'total': 'total5', 'score': 'score5'})
alll = pd.merge(alll, dfs[8], on='name', how='outer').drop(columns=['year', 'region'])
alll = pd.merge(alll, dfs[9], on='name', how='outer').drop(columns=['year', 'title', 'region', 'grade'])
alll = pd.merge(alll, dfs[10], on='name', how='outer').drop(columns=['class_study', 'grade1', 'total2'])
alll

,name,total1,result1,result2,total3,grade2,stage,A,B,C,...,D1_y,A2_y,B2_y,C2_y,D2_y,Sum,place,id_y,title,grade
0,Абдулгужин А.,NaN,NaN,NaN,286.0,10.0,2.0,100.0,43.0,0.0,...,18.0,100.0,100.0,76.0,25.0,530.0,NaN,NaN,NaN,NaN
1,Абдулгужин А.,NaN,NaN,NaN,286.0,10.0,2.0,100.0,43.0,0.0,...,0.0,100.0,31.0,0.0,0.0,247.0,NaN,NaN,NaN,NaN
2,Аболяев В.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,100.0,100.0,47.0,30.0,569.0,NaN,NaN,NaN,NaN
3,Абрамов А.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,100.0,100.0,68.0,10.0,531.0,NaN,NaN,NaN,NaN
4,Абрамов А.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,100.0,100.0,68.0,10.0,531.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3550,Юрчук П.,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,100.0,100.0,39.0,0.0,486.0,NaN,NaN,NaN,NaN
3551,Яковлева П.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,100.0,0.0,0.0,0.0,230.0,NaN,NaN,NaN,NaN
3552,Янковский А.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.0,100.0,47.0,100.0,719.0,NaN,NaN,NaN,NaN
3553,Янковский Д.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18.0,100.0,100.0,100.0,10.0,600.0,NaN,NaN,NaN,NaN


In [79]:
alll = alll.groupby('name', as_index=False).mean()

In [80]:
alll.to_csv('learn_data.csv')

In [81]:
alll.info()

<class 'pandas.DataFrame'>
RangeIndex: 475 entries, 0 to 474
Data columns (total 46 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           475 non-null    str    
 1   total1         16 non-null     float64
 2   result1        16 non-null     float64
 3   result2        15 non-null     float64
 4   total3         198 non-null    float64
 5   grade2         198 non-null    float64
 6   stage          198 non-null    float64
 7   A              198 non-null    float64
 8   B              198 non-null    float64
 9   C              198 non-null    float64
 10  D              198 non-null    float64
 11  E              198 non-null    float64
 12  F              198 non-null    float64
 13  result3        8 non-null      float64
 14  grade3         8 non-null      float64
 15  score4         98 non-null     float64
 16  grade4         284 non-null    float64
 17  A1_x           284 non-null    float64
 18  B1_x           284 no

In [82]:
alll.participation_grade.unique()

AttributeError: 'DataFrame' object has no attribute 'participation_grade'

In [ ]:
dfs[10]